# DataJam 2026 · ConviveData  
## Cuaderno 01 · Limpieza de fuentes originales desde GitHub

Este cuaderno descarga o utiliza las fuentes crudas ubicadas en `data/raw/`, limpia las tres bases principales y exporta archivos limpios a `outputs_limpieza/`.

Fuentes principales:

- Violencia intrafamiliar.
- Lesiones personales.
- Consumo abusivo de SPA.

El procesamiento se hace para Bogotá, 20 localidades, periodo 2018-2025.


## 1. Librerías

In [ ]:
from pathlib import Path
import re
import unicodedata
import urllib.request
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 2. Configuración reproducible de rutas

In [ ]:
# Repositorio público en GitHub Raw.
# Estos enlaces permiten ejecutar el cuaderno desde Colab sin rutas locales.
REPO_RAW = "https://raw.githubusercontent.com/00Jake00/convivedata-vif-bogota-2018-2025-Datajam/main"

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = BASE_DIR / "outputs_limpieza"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVOS_RAW = {
    "vif": {
        "local": RAW_DIR / "violencia_intrafamiliar.csv",
        "url": f"{REPO_RAW}/data/raw/violencia_intrafamiliar.csv",
    },
    "lesiones": {
        "local": RAW_DIR / "lesiones_personales.csv",
        "url": f"{REPO_RAW}/data/raw/lesiones_personales.csv",
    },
    "spa": {
        "local": RAW_DIR / "consumo_abusivo_spa.xlsx",
        "url": f"{REPO_RAW}/data/raw/consumo_abusivo_spa.xlsx",
    },
}

def descargar_si_no_existe(nombre):
    """Usa el archivo local si existe; si no existe, lo descarga desde GitHub Raw."""
    ruta = ARCHIVOS_RAW[nombre]["local"]
    url = ARCHIVOS_RAW[nombre]["url"]

    if not ruta.exists():
        print(f"Descargando {nombre} desde GitHub Raw...")
        urllib.request.urlretrieve(url, ruta)
    else:
        print(f"{nombre}: archivo local encontrado")

    return ruta

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 3. Descarga o localización de fuentes crudas

In [ ]:
ruta_vif = descargar_si_no_existe("vif")
ruta_lesiones = descargar_si_no_existe("lesiones")
ruta_spa = descargar_si_no_existe("spa")

print("Rutas de entrada:")
print("VIF:", ruta_vif)
print("Lesiones:", ruta_lesiones)
print("SPA:", ruta_spa)

## 4. Funciones de lectura y normalización

In [ ]:
def leer_csv_seguro(ruta, sep=";"):
    """
    Lee CSV probando varias codificaciones comunes.
    En estas fuentes el separador esperado es punto y coma.
    """
    for encoding in ["utf-8-sig", "utf-8", "latin1", "cp1252"]:
        try:
            return pd.read_csv(ruta, sep=sep, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo leer el archivo: {ruta}")


def quitar_tildes(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto)
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


ALIAS_LOCALIDAD = {
    "CANDELARIA": "LA CANDELARIA",
    "LA CANDELARIA": "LA CANDELARIA",
    "MARTIRES": "LOS MARTIRES",
    "LOS MARTIRES": "LOS MARTIRES",
    "RAFAEL URIBE": "RAFAEL URIBE URIBE",
    "RAFAEL URIBE URIBE": "RAFAEL URIBE URIBE",
    "CIUDAD BOLIVAR": "CIUDAD BOLIVAR",
    "SAN CRISTOBAL": "SAN CRISTOBAL",
    "USAQUEN": "USAQUEN",
    "FONTIBON": "FONTIBON",
    "ENGATIVA": "ENGATIVA",
    "BOGOTA D.C.": "BOGOTA D.C.",
    "BOGOTA DC": "BOGOTA D.C.",
}

LOCALIDADES_VALIDAS = [
    "USAQUEN", "CHAPINERO", "SANTA FE", "SAN CRISTOBAL", "USME",
    "TUNJUELITO", "BOSA", "KENNEDY", "FONTIBON", "ENGATIVA",
    "SUBA", "BARRIOS UNIDOS", "TEUSAQUILLO", "LOS MARTIRES",
    "ANTONIO NARINO", "PUENTE ARANDA", "LA CANDELARIA",
    "RAFAEL URIBE URIBE", "CIUDAD BOLIVAR", "SUMAPAZ"
]

LOCALIDAD_DISPLAY = {
    "USAQUEN": "Usaquén",
    "CHAPINERO": "Chapinero",
    "SANTA FE": "Santa Fe",
    "SAN CRISTOBAL": "San Cristóbal",
    "USME": "Usme",
    "TUNJUELITO": "Tunjuelito",
    "BOSA": "Bosa",
    "KENNEDY": "Kennedy",
    "FONTIBON": "Fontibón",
    "ENGATIVA": "Engativá",
    "SUBA": "Suba",
    "BARRIOS UNIDOS": "Barrios Unidos",
    "TEUSAQUILLO": "Teusaquillo",
    "LOS MARTIRES": "Los Mártires",
    "ANTONIO NARINO": "Antonio Nariño",
    "PUENTE ARANDA": "Puente Aranda",
    "LA CANDELARIA": "La Candelaria",
    "RAFAEL URIBE URIBE": "Rafael Uribe Uribe",
    "CIUDAD BOLIVAR": "Ciudad Bolívar",
    "SUMAPAZ": "Sumapaz",
}

def normalizar_localidad(valor):
    if pd.isna(valor):
        return np.nan
    texto = quitar_tildes(valor).upper().strip()
    texto = re.sub(r"\s+", " ", texto)
    return ALIAS_LOCALIDAD.get(texto, texto)

def marcar_localizacion(df, col="loc_norm"):
    base = df.copy()
    base["localidad_valida"] = base[col].isin(LOCALIDADES_VALIDAS)
    base["localidad_display"] = base[col].map(LOCALIDAD_DISPLAY)
    return base

## 5. Carga de fuentes originales

In [ ]:
vif_raw = leer_csv_seguro(ruta_vif, sep=";")
lesiones_raw = leer_csv_seguro(ruta_lesiones, sep=";")
spa_raw = pd.read_excel(ruta_spa, sheet_name="in")

resumen_carga = pd.DataFrame([
    {"fuente": "Violencia intrafamiliar", "filas_originales": len(vif_raw), "columnas_originales": len(vif_raw.columns)},
    {"fuente": "Lesiones personales", "filas_originales": len(lesiones_raw), "columnas_originales": len(lesiones_raw.columns)},
    {"fuente": "Consumo abusivo SPA", "filas_originales": len(spa_raw), "columnas_originales": len(spa_raw.columns)},
])

resumen_carga

## 6. Limpieza de VIF y lesiones personales

In [ ]:
def limpiar_base_seguridad(df, hecho_esperado, nombre_fuente):
    base = df.copy()
    base.columns = [str(c).strip() for c in base.columns]

    columnas_requeridas = [
        "SEXO", "ANIO", "MES", "HECHO", "LOCALIDAD", "COD_LOCALIDAD",
        "ARMA_EMPLEADA", "RANGO_VITAL", "RANGO_DEL_DIA", "CANTIDAD"
    ]
    faltantes = [c for c in columnas_requeridas if c not in base.columns]
    if faltantes:
        raise ValueError(f"{nombre_fuente}: faltan columnas requeridas: {faltantes}")

    base["ANIO"] = pd.to_numeric(base["ANIO"], errors="coerce")
    base["MES"] = pd.to_numeric(base["MES"], errors="coerce")
    base["CANTIDAD"] = pd.to_numeric(base["CANTIDAD"], errors="coerce").fillna(0)

    for col in ["HECHO", "SEXO", "RANGO_VITAL", "RANGO_DEL_DIA", "ARMA_EMPLEADA"]:
        base[col] = base[col].astype(str).str.upper().str.strip()

    base["loc_norm"] = base["LOCALIDAD"].apply(normalizar_localidad)
    base = marcar_localizacion(base)

    base = base[base["ANIO"].between(2018, 2025)].copy()
    base = base[base["HECHO"] == hecho_esperado].copy()
    base["fuente"] = nombre_fuente

    columnas = [
        "fuente", "SEXO", "ANIO", "MES", "HECHO", "LOCALIDAD", "loc_norm",
        "localidad_display", "localidad_valida", "COD_LOCALIDAD", "ARMA_EMPLEADA",
        "RANGO_VITAL", "RANGO_DEL_DIA", "CANTIDAD"
    ]
    return base[columnas]


vif_limpia = limpiar_base_seguridad(
    vif_raw,
    hecho_esperado="VIOLENCIA INTRAFAMILIAR",
    nombre_fuente="Violencia intrafamiliar"
)

lesiones_limpia = limpiar_base_seguridad(
    lesiones_raw,
    hecho_esperado="LESIONES PERSONALES",
    nombre_fuente="Lesiones personales"
)

vif_localizada = vif_limpia[vif_limpia["localidad_valida"]].copy()
vif_sin_localizacion = vif_limpia[~vif_limpia["localidad_valida"]].copy()

lesiones_localizada = lesiones_limpia[lesiones_limpia["localidad_valida"]].copy()
lesiones_sin_localizacion = lesiones_limpia[~lesiones_limpia["localidad_valida"]].copy()

print("VIF limpia:", vif_limpia.shape)
print("VIF localizada:", vif_localizada.shape)
print("Casos VIF localizados:", int(vif_localizada["CANTIDAD"].sum()))
print()
print("Lesiones limpia:", lesiones_limpia.shape)
print("Lesiones localizada:", lesiones_localizada.shape)
print("Casos lesiones localizados:", int(lesiones_localizada["CANTIDAD"].sum()))

## 7. Limpieza de consumo abusivo de SPA

In [ ]:
spa_limpia = spa_raw.copy()
spa_limpia.columns = [str(c).strip() for c in spa_limpia.columns]

columnas_requeridas_spa = ["ANO", "MESNOTIFICACION", "NOMBRELOCALIDADRESIDENCIA", "CASOS"]
faltantes_spa = [c for c in columnas_requeridas_spa if c not in spa_limpia.columns]
if faltantes_spa:
    raise ValueError(f"Consumo abusivo SPA: faltan columnas requeridas: {faltantes_spa}")

spa_limpia["ANO"] = pd.to_numeric(spa_limpia["ANO"], errors="coerce")
spa_limpia["MESNOTIFICACION"] = pd.to_numeric(spa_limpia["MESNOTIFICACION"], errors="coerce")
spa_limpia["CASOS"] = pd.to_numeric(spa_limpia["CASOS"], errors="coerce").fillna(0)

spa_limpia["loc_norm"] = spa_limpia["NOMBRELOCALIDADRESIDENCIA"].apply(normalizar_localidad)
spa_limpia = marcar_localizacion(spa_limpia)
spa_limpia["fuente"] = "Consumo abusivo SPA"

spa_limpia = spa_limpia[spa_limpia["ANO"].between(2018, 2025)].copy()

columnas_spa = [
    "fuente", "ANO", "MESNOTIFICACION", "SEXO", "NOMBRELOCALIDADRESIDENCIA",
    "loc_norm", "localidad_display", "localidad_valida", "CURSO_DE_VIDA",
    "TIPOASEGURAMIENTO", "NIVELEDUCATIVO", "NOMBREUPZ", "CASOS"
]
columnas_spa = [c for c in columnas_spa if c in spa_limpia.columns]
spa_limpia = spa_limpia[columnas_spa]

spa_localizada = spa_limpia[spa_limpia["localidad_valida"]].copy()
spa_sin_localizacion = spa_limpia[~spa_limpia["localidad_valida"]].copy()

print("SPA limpia:", spa_limpia.shape)
print("SPA localizada:", spa_localizada.shape)
print("Casos SPA localizados:", int(spa_localizada["CASOS"].sum()))

## 8. Resumen de control

In [ ]:
resumen_limpieza = pd.DataFrame([
    {
        "fuente": "Violencia intrafamiliar",
        "registros_limpios": len(vif_limpia),
        "registros_localizados": len(vif_localizada),
        "registros_no_localizados": len(vif_sin_localizacion),
        "casos_localizados": int(vif_localizada["CANTIDAD"].sum()),
        "periodo": f"{int(vif_limpia['ANIO'].min())}-{int(vif_limpia['ANIO'].max())}",
        "entra_al_visor_actual": "Sí",
    },
    {
        "fuente": "Lesiones personales",
        "registros_limpios": len(lesiones_limpia),
        "registros_localizados": len(lesiones_localizada),
        "registros_no_localizados": len(lesiones_sin_localizacion),
        "casos_localizados": int(lesiones_localizada["CANTIDAD"].sum()),
        "periodo": f"{int(lesiones_limpia['ANIO'].min())}-{int(lesiones_limpia['ANIO'].max())}",
        "entra_al_visor_actual": "Sí",
    },
    {
        "fuente": "Consumo abusivo SPA",
        "registros_limpios": len(spa_limpia),
        "registros_localizados": len(spa_localizada),
        "registros_no_localizados": len(spa_sin_localizacion),
        "casos_localizados": int(spa_localizada["CASOS"].sum()),
        "periodo": f"{int(spa_limpia['ANO'].min())}-{int(spa_limpia['ANO'].max())}",
        "entra_al_visor_actual": "Sí",
    },
])

resumen_limpieza

## 9. Exportación de bases limpias

In [ ]:
salidas = {
    "vif_limpia_localizada_2018_2025.csv": vif_localizada,
    "lesiones_limpia_localizada_2018_2025.csv": lesiones_localizada,
    "spa_abusivo_limpio_localizado_2018_2025.csv": spa_localizada,
    "resumen_limpieza_fuentes.csv": resumen_limpieza,
}

for nombre, df in salidas.items():
    df.to_csv(OUTPUT_DIR / nombre, index=False, encoding="utf-8-sig")
    df.to_csv(PROCESSED_DIR / nombre, index=False, encoding="utf-8-sig")

print("Archivos exportados en:", OUTPUT_DIR)
for archivo in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", archivo.name)

print("\nCopia adicional en:", PROCESSED_DIR)

## 10. Cierre

Este cuaderno deja listas las bases limpias requeridas por el cuaderno de ejecución:

- `vif_limpia_localizada_2018_2025.csv`
- `lesiones_limpia_localizada_2018_2025.csv`
- `spa_abusivo_limpio_localizado_2018_2025.csv`
- `resumen_limpieza_fuentes.csv`
